In [5]:
import pandas as pd
import numpy as np
import torch
from chronos import ChronosPipeline
from tqdm import tqdm


In [6]:

# 1. Load the data you are currently extracting
input_csv = "../data/pixel_level_ET_timeseries.csv"
output_analysis = "../data/subfield_anomaly_results_8mo.csv"

print("Loading pixel-level data...")
df = pd.read_csv(input_csv)


Loading pixel-level data...


In [7]:

# Create a unique ID for every 30m location
df['pixel_id'] = df['field_id'].astype(str) + "_" + df['x'].astype(str) + "_" + df['y'].astype(str)

# 2. Initialize Chronos (Small is usually best for zero-shot ET)
print("Initializing Chronos-T5-Small...")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",  # Switch to "cuda" if you have a GPU
    torch_dtype=torch.float32,
)

# 3. Prepare Sequences
# We need to pivot so each row is one pixel's timeline
# Chronos doesn't care about the 'gap' if we treat the 4 months as a continuous block
pivot_df = df.pivot(index='pixel_id', columns=['year', 'month'], values='relative_ET')
pixel_ids = pivot_df.index.tolist()
series_data = pivot_df.values  # Shape: (Num Pixels, Num Timepoints)

results = []

print(f"Running Chronos on {len(series_data)} pixels...")

# We process in batches to keep the CPU/GPU efficient
batch_size = 64 

for i in tqdm(range(0, len(series_data), batch_size), desc="Zero-Shot Inference"):
    batch = series_data[i : i + batch_size]
    
    # We use all but the last season as 'context' to see if Chronos can predict the most recent values
    # Let's say we reserve the last 8 observations (one full growing season) for testing
    context = torch.tensor(batch[:, :-8], dtype=torch.float32)
    actuals = batch[:, -8:]
    
    # Predict the last 8 months
    with torch.no_grad():
        # num_samples=20 gives us a distribution for uncertainty
        forecast = pipeline.predict(context, prediction_length=8, num_samples=100)
    
    # Forecast shape: [batch, samples, prediction_length]
    forecast_median = np.median(forecast.numpy(), axis=1)
    forecast_std = np.std(forecast.numpy(), axis=1)

    for j in range(len(batch)):
        pixel_idx = i + j
        # Calculate Anomaly Score: How far was the real ET from the median forecast?
        # Residual = Actual - Predicted
        residuals = actuals[j] - forecast_median[j]
        mean_residual = np.mean(np.abs(residuals))
        
        # Persistence: Correlation of the pixel with itself over time
        persistence = np.corrcoef(batch[j, :-1], batch[j, 1:])[0, 1] if np.std(batch[j]) > 0 else 0

        results.append({
            'pixel_id': pixel_ids[pixel_idx],
            'chronos_anomaly_score': mean_residual,
            'forecast_uncertainty': np.mean(forecast_std[j]),
            'temporal_persistence': persistence,
            'is_outlier': mean_residual > (np.mean(forecast_std[j]) * 2) # 2-sigma rule
        })

# 4. Merge back with spatial coordinates and Save
results_df = pd.DataFrame(results)
# Extract x and y back from the pixel_id for mapping
results_df[['field_id', 'x', 'y']] = results_df['pixel_id'].str.split('_', expand=True)

results_df.to_csv(output_analysis, index=False)
print(f"Analysis complete. Results saved to {output_analysis}")

Initializing Chronos-T5-Small...
Running Chronos on 10073 pixels...


Zero-Shot Inference: 100%|██████████| 158/158 [1:35:02<00:00, 36.09s/it]   

Analysis complete. Results saved to ../data/subfield_anomaly_results_8mo.csv
